
Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.
Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:openai/gpt-oss-120b")

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")
    genre: str = Field(description="The genre of the movie")
    train: str = Field(description="The main plot or storyline of the movie")


In [10]:
model_with_structure = model.with_structured_output(Movie, include_raw=True)
model_with_structure


{
  raw: _ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.5'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000011411329100>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000114116EA720>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description

In [4]:
model.invoke("Provide details about the moview Inception")

AIMessage(content='**Inception (2010) – Quick Reference Guide**\n\n| Item | Details |\n|------|---------|\n| **Title** | *Inception* |\n| **Release Year** | 2010 |\n| **Genre** | Science‑fiction, Action, Thriller, Heist |\n| **Running Time** | 148 minutes (≈2\u202fh\u202f28\u202fmin) |\n| **Country** | United States (co‑production with the United Kingdom) |\n| **Language** | English |\n| **Budget** | ≈\u202f$160\u202fmillion (production) |\n| **Box‑Office Gross** | ≈\u202f$836\u202fmillion worldwide |\n| **MPAA Rating** | PG‑13 (Violence, some language, brief drug content) |\n| **Production Companies** | Legendary Pictures, Syncopy Inc. |\n| **Distributor** | Warner Bros. Pictures |\n\n---\n\n### Creative Team\n| Role | Name |\n|------|------|\n| **Director / Co‑Writer / Producer** | Christopher Nolan |\n| **Screenwriters** | Christopher Nolan (original story & script) |\n| **Cinematographer** | Wally Pfister |\n| **Editor** | Lee Smith |\n| **Music Composer** | Hans Zimmer |\n| **Prod

Message output alongside parsed structure

In [11]:
response=model_with_structure.invoke("Provide details about the moview Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants details about the movie Inception. We can provide info: genre, title, plot, release year. Could use the function Movie to output structured data. The function expects genre, title, train (plot), year. So we should call the function with appropriate values.', 'tool_calls': [{'id': 'fc_4d7538c2-6280-43e4-ab6e-90ff54797b76', 'function': {'arguments': '{"genre":"Science Fiction, Thriller","title":"Inception","train":"A skilled thief, Dom Cobb, leads a team to infiltrate the subconscious of a target by entering dreams within dreams, aiming to plant an idea—a process known as inception—while confronting personal demons and navigating layered realities.","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 152, 'prompt_tokens': 160, 'total_tokens': 312, 'completion_time': 0.317762388, 'completion_tokens_details': {'reasoning_tokens': 58}, 'prompt_tim

Nested Structure

In [16]:
class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Michael Caine', role='Professor Stephen Miles')], genres=['Action', 'Adventure', 'Sci-Fi'], budget=160000000.0)

TypedDict

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [18]:
from typing_extensions import TypedDict, Annotated

In [23]:
class MovieDetails(TypedDict):
    """Details about a movie."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The release year of the movie"]
    genres: Annotated[list[str], ..., "List of genres of the movie"]
    budget: Annotated[float | None, None, "Budget in millions USD"]

model_with_typeddict = model.with_structured_output(MovieDetails)

response = model_with_typeddict.invoke("Provide details about the movie Avengers: Endgame")
response

{'budget': 356,
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'Avengers: Endgame',
 'year': 2019}

DataClasses

A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [26]:
from dataclasses import dataclass

In [28]:
@dataclass
class MovieDetails:
    """Details about a movie."""
    title: str
    year: int
    genres: list[str]
    budget: float | None = None


In [29]:
model_with_dataclass = model.with_structured_output(MovieDetails)

In [30]:
response = model_with_dataclass.invoke("Provide details about the movie The Matrix")

In [31]:
response

{'budget': None, 'genres': [], 'title': 'The Matrix', 'year': 1999}